In [10]:
import csv

#i used ai to generate the data
data = [
    [
        ["2024-03-01T09:12:00", "FR7630006000011234567890189", "FR", "BNPPARIBAS", "DE89370400440532013000", "DE", "1250.00", "EUR"],
        ["2024-03-01T10:45:00", "FR7630006000011234567890189", "FR", "BNPPARIBAS", "ES9121000418450200051332", "ES", "7400.50", "EUR"],
        ["2024-03-01T11:20:00", "FR7612345000019876543210189", "FR", "SOCGEN", "IT60X0542811101000000123456", "IT", "300.00", "EUR"],
        ["2024-03-01T14:05:00", "FR7612345000019876543210189", "FR", "SOCGEN", "BE68539007547034", "BE", "6200.00", "EUR"],
        ["2024-03-01T16:30:00", "FR7630006000011234567890189", "FR", "BNPPARIBAS", "NL91ABNA0417164300", "NL", "850.00", "EUR"],
    ],
    [
        ["2024-03-02T08:15:00", "FR7630006000011234567890189", "FR", "BNPPARIBAS", "DE89370400440532013000", "DE", "450.00", "EUR"],
        ["2024-03-02T09:50:00", "FR7611111000012345678901234", "FR", "CREDITMUTUEL", "ES9121000418450200051332", "ES", "900.00", "EUR"],
        ["2024-03-02T13:00:00", "FR7611111000012345678901234", "FR", "CREDITMUTUEL", "IT60X0542811101000000123456", "IT", "3000.00", "EUR"],
        ["2024-03-02T15:40:00", "FR7612345000019876543210189", "FR", "SOCGEN", "BE68539007547034", "BE", "700.00", "EUR"],
        ["2024-03-02T18:10:00", "FR7630006000011234567890189", "FR", "BNPPARIBAS", "NL91ABNA0417164300", "NL", "5600.00", "EUR"],
        ["2024-03-02T19:25:00", "FR7611111000012345678901234", "FR", "CREDITMUTUEL", "DE89370400440532013000", "DE", "125.00", "EUR"],
    ],
    [
        ["2024-03-03T07:45:00", "FR7622222000019876500001234", "FR", "LCL", "ES9121000418450200051332", "ES", "200.00", "EUR"],
        ["2024-03-03T10:10:00", "FR7622222000019876500001234", "FR", "LCL", "BE68539007547034", "BE", "1500.00", "EUR"],
        ["2024-03-03T12:35:00", "FR7612345000019876543210189", "FR", "SOCGEN", "IT60X0542811101000000123456", "IT", "800.00", "EUR"],
        ["2024-03-03T15:00:00", "FR7630006000011234567890189", "FR", "BNPPARIBAS", "NL91ABNA0417164300", "NL", "2500.00", "EUR"],
        ["2024-03-03T17:20:00", "FR7622222000019876500001234", "FR", "LCL", "DE89370400440532013000", "DE", "400.00", "EUR"],
    ],
]


for i in range(len(data)):
    file=open(f"data\\transactions_{i+1}.csv","w")
    writer=csv.writer(file)
    writer.writerow([
        "datetime_transaction",
        "iban_origine",
        "pays_source",
        "banque_source",
        "iban_destinataire",
        "pays_destinataire",
        "montant",
        "devise"
    ])
    for row in data[i]:
        writer.writerow(row)

In [11]:
from decimal import Decimal
from typing import TypedDict
from datetime import datetime


class Transaction(TypedDict) :
    datetime_transaction: datetime
    iban_origine: str
    pays_source: str
    banque_source: str
    iban_destinataire: str
    pays_destinataire: str
    montant: Decimal
    devise: str

def loading_data(path : str):
    transactions : list[Transaction]=[]
    invalid_transactions : int =0

    with open(path, newline="") as file:
            reader = csv.DictReader(file)
            for row in reader:
                try:
                    transaction = Transaction(
                        datetime_transaction=datetime.fromisoformat(row["datetime_transaction"]),
                        iban_origine=row["iban_origine"],
                        pays_source=row["pays_source"],
                        banque_source=row["banque_source"],
                        iban_destinataire=row["iban_destinataire"],
                        pays_destinataire=row["pays_destinataire"],
                        montant=Decimal(row["montant"]),
                        devise=row["devise"],
                    )
                    transactions.append(transaction)
                except :
                    print("something is off with the data entered")
                    invalid_transactions+=1

    return transactions,invalid_transactions

In [12]:
def sum_iban_origine_sent(transactions : list[Transaction])-> dict[str,Decimal]:
    res : dict[str,Decimal]={}
    for transaction in transactions:
        if transaction["iban_origine"] not in res :
            res[transaction["iban_origine"]]=transaction["montant"]

        else : res[transaction["iban_origine"]]+=transaction["montant"]

    return res 


def sum_banque_source_sent(transactions : list[Transaction])-> dict[str,Decimal]:
    res : dict[str,Decimal]={}
    for transaction in transactions:
        if transaction["banque_source"] not in res :
            res[transaction["banque_source"]]=transaction["montant"]

        else : res[transaction["banque_source"]]+=transaction["montant"]

    return res 


def sum_iban_destinataire_received(transactions : list[Transaction])-> dict[str,Decimal]:
    res : dict[str,Decimal]={}
    for transaction in transactions:
        if transaction["iban_destinataire"] not in res :
            res[transaction["iban_destinataire"]]=transaction["montant"]

        else : res[transaction["iban_destinataire"]]+=transaction["montant"]

    return res 


def flag_montant(transactions : list[Transaction], threshold : Decimal=Decimal("5000")) -> list[bool]:
    res : list[bool]=[]

    for transaction in transactions:
        if transaction["montant"]>threshold :
            res.append(True)     
        else : res.append(False)   

    return res   

In [13]:

transactions_1,_ = loading_data(r"data\transactions_1.csv")
transactions_2,_ = loading_data(r"data\transactions_2.csv")
transactions_3,_ = loading_data(r"data\transactions_3.csv")

transactions=transactions_1+transactions_2+transactions_3

In [14]:
# print(len(transactions))

In [15]:
#test
# print(sum_iban_origine_sent(transactions_1))

In [16]:
import sqlite3

def create_table(transactions : list[Transaction], connx : sqlite3.Connection) ->None :
    connx.execute("CREATE TABLE IF NOT EXISTS transactions (" \
    "id INTEGER PRIMARY KEY AUTOINCREMENT," \
    "datetime_transaction TEXT NOT NULL," \
    "iban_origine TEXT NOT NULL," \
    "pays_source TEXT NOT NULL," \
    "banque_source TEXT NOT NULL," \
    "iban_destinataire TEXT NOT NULL," \
    "pays_destinataire TEXT NOT NULL," \
    "montant TEXT NOT NULL," \
    "devise TEXT NOT NULL" \
    ")")

    rows =[(
        transaction["datetime_transaction"],
        transaction["iban_origine"] ,
        transaction["pays_source"],
        transaction["banque_source"],
        transaction["iban_destinataire"],
        transaction["pays_destinataire"],
         str(transaction["montant"]),
          transaction["devise"],

    ) for transaction in transactions]
    try :
        connx.executemany("INSERT INTO transactions (" \
        "datetime_transaction," \
        "iban_origine," \
        "pays_source," \
        "banque_source," \
        "iban_destinataire," \
        "pays_destinataire," \
        "montant," \
        "devise) VALUES (?,?,?,?,?,?,?,?)",rows)

        connx.commit()

    except Exception:
        print("e")
        connx.rollback()


In [17]:
connx = sqlite3.connect("test.db")

create_table(transactions,connx)

connx.close()

C:\Users\dell\AppData\Local\Temp\ipykernel_5788\2809686815.py:28: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  connx.executemany("INSERT INTO transactions (" \


In [20]:
# connx = sqlite3.connect("test.db")
# cursor = connx.execute("SELECT * FROM transactions")

# for row in cursor:
#     print(row)